# Refactored Lecture 03: Iterators & Itertools

## Part 1: Concepts & Mechanics

### 1. Iterators vs. Lists
**The Core Difference:**
* **Lists/Arrays:** Allocate memory for *all* elements at once. Good for random access.
* **Iterators:** Generate values **on demand** (lazy evaluation). Good for large data and memory efficiency.

**Key Iterator Properties:**
1.  **Lazy:** Values are computed one by one.
2.  **Sequential:** Can only move forward.
3.  **One-shot:** Once consumed, they are exhausted.

### 2. Under the Hood: `iter()` and `next()`
An iterator is any object with:
* `__iter__()`: Returns the iterator object itself.
* `__next__()`: Returns the next value. Raises `StopIteration` when finished.

In [1]:
# Create an iterator from a string
it = iter('advanced')

# Manually grab the first few items
print(next(it))  # a
print(next(it))  # d
print(next(it))  # v

# Consuming the REST of the iterator
rest = list(it)
print(f"Rest: {rest}")

# The iterator is now exhausted. Accessing it again creates an empty list.
print(f"Again: {list(it)}")

a
d
v
Rest: ['a', 'n', 'c', 'e', 'd']
Again: []


In [2]:
# Once all values are consumed, calling next(it) or it.__next__() will raise StopIteration
next(it)

StopIteration: 

### 3. Generators (`yield`)
Generators are the easiest way to write iterators.
* Use `yield` instead of `return`.
* The function "pauses" at `yield` and resumes state when `next()` is called.

In [3]:
# Generator function using yield
def demo_xrange(start, stop, step=1):
    while start < stop:
        yield start
        start += step

# Usage
gen = demo_xrange(0, 5)
print(list(gen)) # [0, 1, 2, 3, 4]

[0, 1, 2, 3, 4]


### 4. Comprehensions: List vs. Generator
* **List Comprehension `[]`**: Creates the full list immediately.
* **Generator Expression `()`**: Creates a generator object (lazy).

In [4]:
# List: Uses memory for 10 integers immediately
lst = [i for i in range(10)]
print(f"List: {lst}")

# Generator: No values generated yet
gen = (i for i in range(10))
print(f"Gen: {type(gen)}")
print(f"First value: {next(gen)}")

List: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Gen: <class 'generator'>
First value: 0


## Part 2: The `itertools` Module

### 1. Infinite Iterators
These will run forever unless stopped (e.g., via `break` or `islice`).

| Function | Description | Example |
| :--- | :--- | :--- |
| `count(start, step)` | Infinite counter | 10, 13, 16... |
| `cycle(seq)` | Repeats sequence indefinitely | A, B, C, A, B... |
| `repeat(elem, n)` | Repeats element n times | 'Hello', 'Hello'... |

**Tool: `islice` (Iterator Slice)**
Standard slicing `[start:stop]` doesn't work on iterators. Use `islice`.

In [5]:
from itertools import count, cycle, repeat, islice

# 1. count: Start at 10, step by 3. Take first 5.
print("Count:", list(islice(count(10, 3), 5)))

# 2. cycle: Cycle 'abc'. Take first 7.
print("Cycle:", list(islice(cycle('abc'), 7)))

# 3. repeat: Repeat 'NYU' 3 times.
print("Repeat:", list(repeat('NYU', 3)))

Count: [10, 13, 16, 19, 22]
Cycle: ['a', 'b', 'c', 'a', 'b', 'c', 'a']
Repeat: ['NYU', 'NYU', 'NYU']


### 2. Finite Iterators

#### A. Accumulate & Chain
* `accumulate`: Running totals (or running results of a function).
* `chain`: Flattens multiple iterables into one sequence.

In [6]:
from itertools import accumulate, chain
import operator

# Accumulate (default is add)
data = [1, 2, 3, 4]
print("Sum accumulate:", list(accumulate(data)))                 # [1, 3, 6, 10]
print("Mul accumulate:", list(accumulate(data, operator.mul)))   # [1, 2, 6, 24]

# Chain
list1 = ['a', 'b']
list2 = [1, 2]
print("Chain:", list(chain(list1, list2))) # ['a', 'b', 1, 2]

Sum accumulate: [1, 3, 6, 10]
Mul accumulate: [1, 2, 6, 24]
Chain: ['a', 'b', 1, 2]


#### B. Filtering: Compress, Dropwhile, Takewhile, Filterfalse

| Function | Logic |
| :--- | :--- |
| `compress(data, selectors)` | Keep element if selector is `True`. |
| `takewhile(pred, seq)` | Keep elements **until** predicate is False (then stop). |
| `dropwhile(pred, seq)` | Drop elements **until** predicate is False (then take the rest). |
| `filterfalse(pred, seq)` | Keep elements where predicate is `False`. |

In [7]:
from itertools import compress, dropwhile, takewhile, filterfalse

vals = [6, 7, 8, 1, 2, 3, 9, 10]

# Compress
print("Compress:", list(compress('ABCDEF', [1,0,1,0,1,1]))) # ['A', 'C', 'E', 'F']

# Takewhile (Stops at 1 because 1 > 5 is False)
print("Takewhile >5:", list(takewhile(lambda x: x > 5, vals))) # [6, 7, 8]

# Dropwhile (Skips 6,7,8. Starts taking at 1. Takes EVERYTHING after that)
print("Dropwhile >5:", list(dropwhile(lambda x: x > 5, vals))) # [1, 2, 3, 9, 10]

# Filterfalse (Only keeps small numbers)
print("Filterfalse >5:", list(filterfalse(lambda x: x > 5, vals))) # [1, 2, 3]

Compress: ['A', 'C', 'E', 'F']
Takewhile >5: [6, 7, 8]
Dropwhile >5: [1, 2, 3, 9, 10]
Filterfalse >5: [1, 2, 3]


#### C. Groupby
Groups consecutive elements with the same key.
* **Important:** The input sequence **must be sorted** by the grouping key first!

In [8]:
from itertools import groupby
from collections.abc import Iterator

# Example: Group numbers by quotient of 5 (0-4 group 0, 5-9 group 1...)
numbers = range(15)

print("Groupby:")
for key, group in groupby(numbers, lambda x: x // 5):
    print("group is an Iterator object: ", isinstance(group, Iterator))
    print(f"Key {key}: {list(group)}")

Groupby:
group is an Iterator object:  True
Key 0: [0, 1, 2, 3, 4]
group is an Iterator object:  True
Key 1: [5, 6, 7, 8, 9]
group is an Iterator object:  True
Key 2: [10, 11, 12, 13, 14]


#### D. Starmap, Tee, Zip_longest
* `starmap`: Like `map()`, but unpacks arguments from tuples.
* `tee`: Splits one iterator into `n` independent iterators.
* `zip_longest`: Zips lists of uneven length (fills missing values).

In [9]:
from itertools import starmap, tee, zip_longest

# Starmap: computes pow(2,5), pow(3,2), pow(10,3)
pairs = [(2, 5), (3, 2), (10, 3)]
print("Starmap (pow):", list(starmap(pow, pairs)))

# Tee: Create 2 independent iterators from one data source
iter1, iter2 = tee("ABC", 2)
print(f"Tee 1: {list(iter1)}")
print(f"Tee 2: {list(iter2)}")

# Zip Longest
print("Zip Longest:", list(zip_longest('AB', '1234', fillvalue='-')))

Starmap (pow): [32, 9, 1000]
Tee 1: ['A', 'B', 'C']
Tee 2: ['A', 'B', 'C']
Zip Longest: [('A', '1'), ('B', '2'), ('-', '3'), ('-', '4')]


## Part 3: Combinatoric Generators

| Function | Concept | Order Matters? | Repeats? |
| :--- | :--- | :--- | :--- |
| `product` | Cartesian (Nested Loops) | Yes | Yes |
| `permutations` | All possible orderings | Yes | No |
| `combinations` | Unique groups | No | No |
| `combinations_with_replacement` | Unique groups + repeats | No | Yes |

In [10]:
from itertools import product, permutations, combinations, combinations_with_replacement

letters = 'ABC'

print("Product (AxB):", list(product(letters, repeat=2)))
# ('A', 'A'), ('A', 'B')... ('C', 'C')

print("Permutations (Order matters):", list(permutations(letters, 2)))
# ('A', 'B'), ('B', 'A')...

print("Combinations (Unique pairs):", list(combinations(letters, 2)))
# ('A', 'B'), ('A', 'C'), ('B', 'C') - No ('B', 'A')

print("Combos w/ Replace:", list(combinations_with_replacement(letters, 2)))
# ('A', 'A'), ('A', 'B')...

Product (AxB): [('A', 'A'), ('A', 'B'), ('A', 'C'), ('B', 'A'), ('B', 'B'), ('B', 'C'), ('C', 'A'), ('C', 'B'), ('C', 'C')]
Permutations (Order matters): [('A', 'B'), ('A', 'C'), ('B', 'A'), ('B', 'C'), ('C', 'A'), ('C', 'B')]
Combinations (Unique pairs): [('A', 'B'), ('A', 'C'), ('B', 'C')]
Combos w/ Replace: [('A', 'A'), ('A', 'B'), ('A', 'C'), ('B', 'B'), ('B', 'C'), ('C', 'C')]


### Real World Example: Product for Matrix Indexing
`product` is often used to avoid deep nested loops.

In [11]:
import numpy as np
from itertools import product

# Fill a 3x3x3 grid
M = np.zeros((3, 3, 3))
indices = product(range(3), range(3), range(3)) # Generates (0,0,0), (0,0,1)...

for i, idx in enumerate(indices):
    M[idx] = i * 10

print("Matrix Shape:", M.shape)
print("Value at (0,0,1):", M[0,0,1])

Matrix Shape: (3, 3, 3)
Value at (0,0,1): 10.0


In [12]:
M

array([[[  0.,  10.,  20.],
        [ 30.,  40.,  50.],
        [ 60.,  70.,  80.]],

       [[ 90., 100., 110.],
        [120., 130., 140.],
        [150., 160., 170.]],

       [[180., 190., 200.],
        [210., 220., 230.],
        [240., 250., 260.]]])